In [3]:
import logging
from groq import RateLimitError, APIConnectionError, APITimeoutError
from pydantic import ValidationError
import os
import time
from groq import Groq
from pydantic import BaseModel
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)
class Advice(BaseModel):
    answer: str
    topic: str
    difficulty: str
history = []
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)
system_prompt = {
    "role":"system",
    "content": """you are a tutor for school going kids. your ans should be short and easy to understand.You are a tutor for school-going kids.
    Return your response ONLY as JSON. The JSON must have exactly these fields: answer: string, topic: string, difficulty: string
    Do not add any other fields. Example:
    {
        "answer": "Gravity is a force that pulls objects toward each other.",
        "topic": "Physics",
        "difficulty": "Easy"
    }"""
}
history.append(system_prompt)
while(True):
    user_input = input("Enter your query ...")
    if user_input.strip() == "":
        print("Please enter a valid question...")
        continue
    if ["stop", "exit", "break", "quit"].__contains__(user_input.strip().lower()):
        break
    elif user_input.strip().lower() == "reset":
        history = []
        history.append(system_prompt)
        continue
    history.append({
        "role": "user",
        "content": user_input
    })
    try:   
        chat_completion = client.chat.completions.create(
            messages=history,
            model="llama-3.3-70b-versatile",
            stream=True
        )
        ai_ans=""
        for chunk in chat_completion:
            is_present = chunk.choices[0].delta.content == None
            if is_present == False:
                print(chunk.choices[0].delta.content, end="", flush=True)
                ai_ans += chunk.choices[0].delta.content
                time.sleep(0.15)
        try:
            advice = Advice.model_validate_json(ai_ans)
        except ValidationError as e:
            print("AI response is not following expected model structure.")
        history.append({
            "role":"assistant",
            "content" : ai_ans
        })
    except KeyboardInterrupt as e:
        logger.info("Chatbot stopped...")
        break
    except APIConnectionError as e:
        logger.error("Connection can't be established.")
    except RateLimitError as e:
        logger.error("Too many requests. Please wait...")
    except APITimeoutError as e:
        logger.error("It is taking too long...")
    except Exception as e:
        logger.error(f"Error{e}")
    

2026-08-15 14:24:29,332 | INFO | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


{
    "answer": "Gravity is a force that pulls objects towards each other.",
    "topic": "Physics",
    "difficulty": "Easy"
}Data is fineanswer='Gravity is a force that pulls objects towards each other.' topic='Physics' difficulty='Easy'
